In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv

load_dotenv()

In [2]:
# 1. Define State 
class BatsmanState(TypedDict):
    runs: int
    balls: int 
    fours: int
    sixes: int

    strick_rate: float
    balls_per_boundry: float 
    boundry_perc: float 
    summry: str

In [3]:
# In parallel nodes, return ONLY the updated keys to prevent InvalidUpdateError
def cal_strick_rate(state: BatsmanState):
    strick_rate = (state['runs'] / state['balls']) * 100
    return {'strick_rate': strick_rate}

In [4]:
def cal_balls_per_boundry(state: BatsmanState):
    total_boundaries = state['fours'] + state['sixes']
    balls_per_boundry = state['balls'] / total_boundaries if total_boundaries > 0 else 0.0
    return {'balls_per_boundry': balls_per_boundry}

In [5]:
def cal_boundry_perc(state: BatsmanState):
    boundary_runs = (state['fours'] * 4) + (state['sixes'] * 6)
    boundry_perc = (boundary_runs / state['runs']) * 100 if state['runs'] > 0 else 0.0
    return {'boundry_perc': boundry_perc}

In [6]:
def summry(state: BatsmanState):
    summary_text = f"""
    --- Batsman Performance Summary ---
    Strike Rate        : {state['strick_rate']:.2f}
    Balls Per Boundary : {state['balls_per_boundry']:.2f}
    Boundary Runs %    : {state['boundry_perc']:.2f}%
    """
    return {'summry': summary_text}

In [7]:
# 2. Build Graph
graph = StateGraph(BatsmanState)

# Add nodes
graph.add_node('cal_strick_rate', cal_strick_rate)
graph.add_node('cal_balls_per_boundry', cal_balls_per_boundry)
graph.add_node('cal_boundry_perc', cal_boundry_perc)
graph.add_node('summry', summry)

# Add parallel edges from START
graph.add_edge(START, 'cal_strick_rate')
graph.add_edge(START, 'cal_balls_per_boundry')
graph.add_edge(START, 'cal_boundry_perc')

# Fan-in edges to summary node
graph.add_edge('cal_strick_rate', 'summry')
graph.add_edge('cal_balls_per_boundry', 'summry')
graph.add_edge('cal_boundry_perc', 'summry')

# Edge to END
graph.add_edge('summry', END)

# 3. Compile graph into runnable workflow
workflow = graph.compile()

In [8]:
# 4. Invoke Workflow
initial_state = {
    'runs': 100, 
    'balls': 50,
    'fours': 6,
    'sixes': 4
}

result = workflow.invoke(initial_state)
print(result['summry'])